In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path


@dataclass
class DocumentJob:
    """
    Background job for processing an uploaded document.
    """

    material_id: str
    file_path: str


class DocumentWorker:
    """
    Executes the document-processing pipeline.

    Pipeline:

        uploaded PDF
             ↓
        text extraction
             ↓
        cleaning
             ↓
        chunking
             ↓
        Local MiniLM embeddings
             ↓
        MongoDB Atlas Vector Search storage
    """

    def __init__(self, database):
        self.database = database

    def process(
        self,
        job: DocumentJob,
    ) -> dict:

        from app.services.document_service import (
            DocumentService,
        )

        from app.services.ai_service import (
            AIService,
        )

        materials = self.database.collection(
            "materials"
        )

        material = materials.find_one(
            {
                "id": job.material_id,
            },
            {
                "_id": 0,
            },
        )

        if material is None:
            raise ValueError(
                f"Material not found: {job.material_id}"
            )

        file_path = Path(
            job.file_path
        )

        if not file_path.exists():
            raise FileNotFoundError(
                f"Document file not found: {file_path}"
            )

        if file_path.suffix.lower() != ".pdf":
            raise ValueError(
                "DocumentWorker currently supports PDF files only."
            )

        ai_service = AIService(
            database=self.database
        )

        service = DocumentService(
            database=self.database
        )

        return service.process_material(
            material=material,
            file_path=file_path,
            ai_service=ai_service,
        )